# Ensemble Learning on Sri Lankan Used Car Prices
**Dataset:** Sri Lankan Vehicle (Car) Price Dataset (ikman.lk & riyasewana.lk), Kaggle

**Goal:** predict used car price (in lakhs) and compare ensemble methods:
- Baseline: single Decision Tree
- Bagging: Random Forest
- Boosting: XGBoost
- Stacking: Random Forest + XGBoost → Ridge
- Hybrid: Random Forest + ANN

Run top to bottom in Google Colab. Upload `car_price_dataset.csv` when asked.

## 1. Setup

In [ ]:
!pip install -q xgboost

import time, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
RS = 42

In [ ]:
# Upload the CSV (Colab). If running locally, just set the path.
try:
    from google.colab import files
    up = files.upload()
    PATH = list(up.keys())[0]
except ImportError:
    PATH = "car_price_dataset.csv"

df = pd.read_csv(PATH)
print(df.shape)
df.head()

## 2. Data understanding

In [ ]:
df.info()
print("\nMissing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.nunique()

In [ ]:
df["Price"].describe()  # Price is in lakhs (1 lakh = Rs 100,000)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Price"], bins=60, ax=ax[0]).set_title("Price (lakhs), right-skewed")
sns.histplot(np.log1p(df["Price"]), bins=60, ax=ax[1]).set_title("log(1 + Price), close to normal")
plt.tight_layout(); plt.show()

### Data quality finding: Mileage is derived from Year
Mileage has exactly as many unique values as YOM and a correlation of -1.0 with it.
It looks like it was filled in as roughly 11,000 km per year of age, not real odometer readings.
So it carries **no extra information** and is dropped.

In [ ]:
print("Unique YOM:", df["YOM"].nunique(), "| Unique Mileage:", df["Millage(KM)"].nunique())
print("Correlation YOM vs Mileage:", round(df["YOM"].corr(df["Millage(KM)"]), 6))
plt.figure(figsize=(6, 4))
plt.scatter(df["YOM"], df["Millage(KM)"], s=5)
plt.xlabel("Year of manufacture"); plt.ylabel("Mileage (km)"); plt.title("Mileage is a straight line of YOM")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.groupby("Brand")["Price"].median().sort_values(ascending=False).head(15).plot.bar(ax=ax[0], title="Median price by brand (top 15)")
df.groupby("YOM")["Price"].median().plot(ax=ax[1], title="Median price by year of manufacture")
plt.tight_layout(); plt.show()

### Relationships: engine size and town vs price

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=df, x="Engine (cc)", y="Price", hue="Fuel Type", alpha=0.5, s=12, ax=ax[0])
ax[0].set_yscale("log"); ax[0].set_title("Engine capacity vs price (log scale)")
print("Correlation Engine (cc) vs log(Price):", round(df["Engine (cc)"].corr(np.log1p(df["Price"])), 3))

top_towns = df["Town"].value_counts().head(12).index
tdf = df[df["Town"].isin(top_towns)]
order = tdf.groupby("Town")["Price"].median().sort_values(ascending=False).index
sns.boxplot(data=tdf, x="Town", y="Price", order=order, showfliers=False, ax=ax[1])
ax[1].set_yscale("log"); ax[1].set_title("Price by town (12 towns with the most ads)")
ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Why is the engine vs price correlation so weak? Big engines are mostly old diesel vehicles.
print(df.groupby("Fuel Type")[["Engine (cc)", "YOM", "Price"]].median())
# Remove the effect of year from both, then correlate (partial correlation)
lp = np.log1p(df["Price"])
res_p = lp - np.polyval(np.polyfit(df["YOM"], lp, 1), df["YOM"])
res_e = df["Engine (cc)"] - np.polyval(np.polyfit(df["YOM"], df["Engine (cc)"], 1), df["YOM"])
print("Partial correlation Engine vs log(Price), controlling for year:", round(np.corrcoef(res_e, res_p)[0, 1], 3))

**Findings:**
- Engine size alone looks almost unrelated to price (correlation about 0.06), because most big-engine cars are old diesels. Comparing cars of the same year, bigger engines **do** cost more (partial correlation about 0.53). Year is a *confounder* here.
- Dehiwala-Mount Lavinia, Colombo and Negombo have the highest median prices, Kurunegala and Gampaha the lowest. The ranges overlap a lot, so location matters far less than brand, year and engine size.

## 3. Preprocessing

In [ ]:
data = df.drop(columns=["Unnamed: 0", "Date", "Millage(KM)"])
data = data.drop_duplicates().reset_index(drop=True)
data["Age"] = 2025 - data["YOM"]

TARGET = "Price"
HIGH_CARD = ["Brand", "Model", "Town"]                       # many categories -> target encoding
LOW_CARD = ["Gear", "Fuel Type", "Leasing", "Condition",
            "AIR CONDITION", "POWER STEERING", "POWER MIRROR", "POWER WINDOW"]  # few categories -> one-hot
NUMERIC = ["YOM", "Age", "Engine (cc)"]
FEATURES = HIGH_CARD + LOW_CARD + NUMERIC

X = data[FEATURES]
y = np.log1p(data[TARGET])   # train on log price, convert back for metrics

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RS)
print("Train:", X_train.shape, " Test:", X_test.shape)

def make_preprocessor(scale=False):
    num = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("te", TargetEncoder(random_state=RS), HIGH_CARD),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False), LOW_CARD),
        ("num", num, NUMERIC),
    ])

## 4. Train the models: baseline, bagging, boosting

In [ ]:
models = {
    "Decision Tree (baseline)": DecisionTreeRegressor(random_state=RS),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=RS),
    "XGBoost": XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=RS),
}
FAMILY = {"Decision Tree (baseline)": "Single model", "Random Forest": "Bagging", "XGBoost": "Boosting"}

def evaluate(name, y_true_log, y_pred_log, train_time, family):
    yt, yp = np.expm1(y_true_log), np.expm1(y_pred_log)
    return {"Model": name, "Family": family,
            "R2": r2_score(yt, yp), "MAE (lakhs)": mean_absolute_error(yt, yp),
            "RMSE (lakhs)": np.sqrt(mean_squared_error(yt, yp)),
            "MAPE (%)": np.mean(np.abs((yt - yp) / yt)) * 100, "Train time (s)": train_time}

results, fitted, preds = [], {}, {}
for name, model in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    t0 = time.time(); pipe.fit(X_train, y_train); t = time.time() - t0
    p = pipe.predict(X_test)
    results.append(evaluate(name, y_test, p, t, FAMILY[name]))
    fitted[name], preds[name] = pipe, p
    print(f"{name:26s} R2={results[-1]['R2']:.3f}  MAE={results[-1]['MAE (lakhs)']:.2f}  time={t:.1f}s")

## 5. Stacking: Random Forest + XGBoost → Ridge

**How it works:**
1. Two base models are trained: Random Forest (bagging) and XGBoost (boosting).
2. `StackingRegressor` creates their training-set predictions with **5-fold cross-validation** (out-of-fold), so the meta-model never sees predictions on cars a base model was trained on.
3. A **Ridge regression** meta-model learns the best weighted combination of the four predictions.

RF and XGBoost make different kinds of mistakes, so combining them can cancel some errors.

In [ ]:
stack = StackingRegressor(
    estimators=[
        ("rf", RandomForestRegressor(n_estimators=300, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=RS)),
        ("xgb", XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=RS)),
    ],
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)), cv=5)

pipe = Pipeline([("prep", make_preprocessor()), ("model", stack)])
t0 = time.time(); pipe.fit(X_train, y_train); t = time.time() - t0
p = pipe.predict(X_test)
results.append(evaluate("Stacking", y_test, p, t, "Stacking"))
fitted["Stacking"], preds["Stacking"] = pipe, p
r = results[-1]
print(f"Stacking  R2={r['R2']:.3f}  MAE={r['MAE (lakhs)']:.2f}  time={t:.1f}s")

stack_weights = dict(zip(["Random Forest", "XGBoost"],
                         pipe.named_steps["model"].final_estimator_.coef_.round(3)))
print("Meta-model weights:", stack_weights)

## 6. Hybrid model: Random Forest + ANN (a stacking variant)

**How it works (stacking):**
1. Random Forest is trained and makes *out-of-fold* predictions on the training set (so the ANN never sees leaked predictions).
2. The ANN (MLP neural network) gets the **original features + the RF prediction** as input.
3. RF captures the tree-style rules (brand, year thresholds), the ANN learns smooth corrections on top.

In [ ]:
prep_ann = make_preprocessor(scale=True)
Xtr_p = prep_ann.fit_transform(X_train, y_train)
Xte_p = prep_ann.transform(X_test)

rf_base = RandomForestRegressor(n_estimators=300, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=RS)
t0 = time.time()
rf_oof = cross_val_predict(rf_base, Xtr_p, y_train, cv=KFold(5, shuffle=True, random_state=RS), n_jobs=-1)
rf_base.fit(Xtr_p, y_train)
rf_test = rf_base.predict(Xte_p)

# ANN on its own (for comparison)
ann = MLPRegressor(hidden_layer_sizes=(128, 64), early_stopping=True, max_iter=500, random_state=RS)
t_a = time.time(); ann.fit(Xtr_p, y_train); t_ann = time.time() - t_a
p_ann = ann.predict(Xte_p)

# Hybrid: ANN with RF prediction as an extra feature
hybrid_ann = MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=500, random_state=RS)
hybrid_ann.fit(np.column_stack([Xtr_p, rf_oof]), y_train)
p_hybrid = hybrid_ann.predict(np.column_stack([Xte_p, rf_test]))
t_h = time.time() - t0

results.append(evaluate("ANN (alone)", y_test, p_ann, t_ann, "Neural network"))
results.append(evaluate("Hybrid RF + ANN", y_test, p_hybrid, t_h, "Hybrid"))
preds["ANN (alone)"], preds["Hybrid RF + ANN"] = p_ann, p_hybrid
for r in results[-2:]:
    print(f"{r['Model']:26s} R2={r['R2']:.3f}  MAE={r['MAE (lakhs)']:.2f}")

## 7. Results

In [ ]:
res = pd.DataFrame(results).sort_values("MAE (lakhs)").reset_index(drop=True)
res.style.format({"R2": "{:.3f}", "MAE (lakhs)": "{:.2f}", "RMSE (lakhs)": "{:.2f}",
                  "MAPE (%)": "{:.1f}", "Train time (s)": "{:.1f}"}).background_gradient(subset=["MAE (lakhs)"], cmap="RdYlGn_r")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
r = res.sort_values("MAE (lakhs)", ascending=False)
ax[0].barh(r["Model"], r["MAE (lakhs)"]); ax[0].set_title("Mean absolute error (lower is better)"); ax[0].set_xlabel("lakhs")
ax[1].barh(r["Model"], r["R2"]); ax[1].set_title("R² (higher is better)"); ax[1].set_xlim(0.7, 1)
plt.tight_layout(); plt.show()

### RF vs XGBoost head to head

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, name in zip(ax, ["Random Forest", "XGBoost"]):
    yt, yp = np.expm1(y_test), np.expm1(preds[name])
    a.scatter(yt, yp, s=6, alpha=0.5); m = yt.max()
    a.plot([0, m], [0, m], "r--"); a.set_xscale("log"); a.set_yscale("log")
    a.set_title(name); a.set_xlabel("Actual (lakhs)"); a.set_ylabel("Predicted (lakhs)")
plt.tight_layout(); plt.show()
res[res.Model.isin(["Random Forest", "XGBoost"])]

### Feature importance (RF vs XGBoost)

In [ ]:
def importances(name):
    pipe = fitted[name]
    names = pipe.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=names)
    # merge one-hot columns back to their original feature
    orig = [n.split("__", 1)[1] for n in names]
    orig = [next((f for f in FEATURES if o == f or o.startswith(f + "_")), o) for o in orig]
    return imp.groupby(orig).sum().sort_values(ascending=False)

fi = pd.DataFrame({"Random Forest": importances("Random Forest"), "XGBoost": importances("XGBoost")})
fi = fi / fi.sum()
fi.sort_values("Random Forest").plot.barh(figsize=(8, 6), title="Normalised feature importance")
plt.show()
fi.sort_values("Random Forest", ascending=False)

## 8. Save everything for the dashboard

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)
res.to_csv("artifacts/results.csv", index=False)
fi.to_csv("artifacts/feature_importance.csv")
pd.DataFrame({"actual": np.expm1(y_test.values),
              **{k: np.expm1(v) for k, v in preds.items()}}).to_csv("artifacts/test_predictions.csv", index=False)

json.dump({k: float(v) for k, v in stack_weights.items()}, open("artifacts/stacking_weights.json", "w"))
for name in ["Random Forest", "XGBoost", "Stacking"]:
    joblib.dump(fitted[name], f"artifacts/{name.replace(' ', '_').lower()}.pkl", compress=3)
joblib.dump({"prep": prep_ann, "rf": rf_base, "ann": hybrid_ann}, "artifacts/hybrid_rf_ann.pkl", compress=3)

# options for the dashboard input form
meta = {"brand_models": data.groupby("Brand")["Model"].unique().apply(sorted).apply(list).to_dict(),
        "towns": sorted(data["Town"].unique()), "low_card": {c: sorted(data[c].unique()) for c in LOW_CARD},
        "yom_range": [int(data.YOM.min()), int(data.YOM.max())], "features": FEATURES}
json.dump(meta, open("artifacts/meta.json", "w"))
data.to_csv("artifacts/clean_data.csv", index=False)

!zip -qr artifacts.zip artifacts
try:
    files.download("artifacts.zip")
except NameError:
    print("Saved to artifacts.zip")

## 9. Research: RF vs Gradient Boosting

| | Random Forest (bagging) | Gradient Boosting (XGBoost / LightGBM / CatBoost) |
|---|---|---|
| How trees are built | Independently, in parallel, on bootstrap samples | Sequentially, each tree fixes the errors of the previous ones |
| Main thing it reduces | Variance | Bias (and variance with regularisation) |
| Tuning effort | Low, good results with defaults | Higher (learning rate, depth, n_estimators) |
| Overfitting risk | Low | Higher if not regularised / early stopped |
| Speed | Parallel, fast to train | XGBoost moderate, LightGBM fastest, CatBoost slower but strong on categoricals |
| Categorical data | Needs encoding | CatBoost handles natively |

**Tabular vs image/text data**
- On **tabular** data (like this one) tree ensembles are usually the best choice: they handle mixed feature types, need little preprocessing and often beat neural networks.
- On **images and text**, tree ensembles struggle: they can't learn spatial patterns (pixels) or word order/meaning. CNNs and transformers are far better there. Trees can still be useful **on top of extracted features** (e.g. CNN embeddings or TF-IDF).

**Conclusion:** fill in from the results table above (which model won, by how much, and the speed trade-off).